# M1. 데이터 확인 (01_data_check)

`data/raw`의 원본 6개 파일을 읽어 **구조 · 결측 · 중복 · 기준연도 · 핵심 컬럼**을 확인합니다.
- 원본은 수정하지 않고 읽기만 합니다.
- 위에서부터 순서대로 실행하면 됩니다. 결과 요약은 `data/analysis/01_data_check_summary.csv`로 저장됩니다.

## 1. 준비: 프로젝트 경로와 파일 목록

In [1]:
from pathlib import Path
import re
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# 노트북 위치와 상관없이 data/raw가 있는 폴더를 위로 올라가며 찾는다
ROOT = Path.cwd()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
assert RAW.exists(), "data/raw 폴더를 찾지 못했습니다. 프로젝트 폴더에서 노트북을 여세요."

print("프로젝트 폴더:", ROOT)
for p in sorted(RAW.iterdir()):
    if p.is_file() and not p.name.startswith("."):
        print(f"  {p.name}  ({p.stat().st_size/1024:,.0f} KB)")

프로젝트 폴더: /Users/song/Desktop/unit1_project
  1_한국산업안전보건공단_산업재해 고위험요인(SIF) 아카이브_20260401.xlsx  (1,398 KB)
  2_한국산업안전보건공단_산업중분류별 규모별 사고재해자수_20251231.csv  (2 KB)
  3_한국산업안전보건공단_규모별 사고사망자수_20251231.csv  (1 KB)
  4_한국산업안전보건공단_산업중분류별 규모별 사업장수_20251231.csv  (2 KB)
  5_한국산업안전보건공단_산업중분류별 규모별 사망만인율_20251231.csv  (2 KB)
  6_고용노동부_산업재해 사고정보 예방조치 현황_AI친화 학습데이터_20260917.csv  (2 KB)


## 2. 6개 파일 읽기
- 1번(xlsx): `개요` 시트는 비어 있고, 표는 두 시트에 있으며 머리글 위치가 다릅니다. 위치를 지정해서 읽습니다.
- 2~5번(CSV): cp949, 6번(CSV): utf-8-sig. 두 인코딩을 차례로 시도합니다.

In [3]:
def find(num):
    """파일명이 '번호_'로 시작하는 CSV 1개를 찾는다."""
    hits = [p for p in RAW.iterdir() if p.is_file() and p.suffix.lower() == ".csv"
            and p.name.startswith(f"{num}_")]
    assert len(hits) == 1, f"{num}번 파일이 {len(hits)}개입니다: {[h.name for h in hits]}"
    return hits[0]

def csv_auto(path):
    for enc in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩을 읽지 못했습니다: {path.name}")

xlsx = [p for p in RAW.iterdir() if p.suffix.lower() in (".xlsx", ".xls") and not p.name.startswith(("~", "."))]
assert len(xlsx) == 1, f"엑셀 파일이 {len(xlsx)}개입니다: {[x.name for x in xlsx]}"
x1 = xlsx[0]

# 1번: 제조업등 시트(머리글 3번째 줄), 건설업 시트(머리글 줄이 따로 없어 열 이름 직접 지정)
mf = pd.read_excel(x1, sheet_name="아카이브(제조업등)", header=2, usecols="B:J")
mf.columns = [str(c).replace("\n", "").replace(" ", "") for c in mf.columns]
cs = pd.read_excel(x1, sheet_name="아카이브(건설업)", header=None, skiprows=4, usecols="B:J")
cs.columns = ["연번", "공종", "작업명", "단위작업명", "재해종류", "재해개요", "기인물", "재해유발요인", "위험성감소대책(예시)"]

dfs = {
    "1a_SIF_제조업등": mf,
    "1b_SIF_건설업": cs,
    "2_업종별규모별_재해자수": csv_auto(find(2)),
    "3_규모별_사망자수": csv_auto(find(3)),
    "4_업종별규모별_사업장수": csv_auto(find(4)),
    "5_업종별규모별_사망만인율": csv_auto(find(5)),
    "6_출처목록": csv_auto(find(6)),
}
for k, d in dfs.items():
    print(f"{k:<26} {d.shape[0]:>6,} 행 × {d.shape[1]:>2} 열")

1a_SIF_제조업등                 2,573 행 ×  9 열
1b_SIF_건설업                  3,459 행 ×  9 열
2_업종별규모별_재해자수                  30 행 × 12 열
3_규모별_사망자수                     10 행 × 23 열
4_업종별규모별_사업장수                  30 행 × 12 열
5_업종별규모별_사망만인율                 30 행 × 12 열
6_출처목록                          5 행 ×  4 열


## 3. 파일별 구조 확인 (shape · columns · head)

In [4]:
for name, d in dfs.items():
    print("=" * 90)
    print(f"[{name}]  {d.shape[0]:,}행 × {d.shape[1]}열")
    print("컬럼:", list(d.columns))
    display(d.head(3))

[1a_SIF_제조업등]  2,573행 × 9열
컬럼: ['연번', '산재업종(대분류)', '산재업종(중분류)', '산재업종(소분류)', '재해개요', '기인물', '고위험작업·상황', '재해유발요인', '위험성감소대책(예시)']


,연번,산재업종(대분류),산재업종(중분류),산재업종(소분류),재해개요,기인물,고위험작업·상황,재해유발요인,위험성감소대책(예시)
0,1,제조업,화학및고무제품제조업,화학비료제조업,2019년 10월경 OO사업장에서 재해자가 화물용 리프트에 제품을 싣는 작업 중 리프트와 작업장 사이...,분류 불가,리프트·승강기 관련 작업,『화물용 승강기 또는 리프트(기계식 주차설비 포함)의 운반구(기)에서 작업 또는 이동』하다가 승강로 ...,▶ 개구부에 추락방지용 덮개 설치\n▶ 카(운반구)에 울 추가 설치\n▶ PLC 및 각 층 위치센서를...
1,2,제조업,화학및고무제품제조업,화학비료제조업,2017년 11월경 OO사업장 음식물 이송 작업장에서 재해자가 작동중인 컨베이어 점검 중 벨트 사이에...,컨베이어,비정형 작업(점검),"『설비에 안전장치가 설치되어 있지 않거나, 불량하거나, 설치된 안전장치를 해제한 상태』로 사용 중 사...",▶ 컨베이어 벨트와 풀리 사이에 방호울 설치\n▶ 컨베이어 비상정지장치 설치
2,3,제조업,화학및고무제품제조업,화학비료제조업,2018년 6월경 OO사업장 혼합 공정에서 재해자가 혼합기 내부에서 점검 작업 중 동료 근로자가 혼합...,혼합기,비정형 작업(점검),『정비·보수·청소·교체·조정·점검 등의 작업 중 작업자 외 다른 사람이 설비를 가동』하여 갑자기 작동...,▶ 키 전환스위치 설치\n▶ 혼합기 덮개 연동장치 설치\n▶ 비정형(점검·청소·정비) 작업 수행 시 ...


[1b_SIF_건설업]  3,459행 × 9열
컬럼: ['연번', '공종', '작업명', '단위작업명', '재해종류', '재해개요', '기인물', '재해유발요인', '위험성감소대책(예시)']


,연번,공종,작업명,단위작업명,재해종류,재해개요,기인물,재해유발요인,위험성감소대책(예시)
0,1,1. 토공사,1.1 굴착 작업,1.1.1 굴착 장비반입,추락,2019년 03월경 ○○ 현장 1층에서 피재자가 장비(미니 백호 1ton)진입을 위하여 해당 방호시설...,바닥개구부(자재인양구 등),굴착기 진입을 위해 현장 정리 중 주차타워 상부 개구부를 인식하지 못하고 추락,▶ 추락할 위험이 있는 개구부에는 안전난간 또는 덮개 등을 충분한 강도를 가진 구조로 튼튼하게 설치\...
1,2,1. 토공사,1.1 굴착 작업,1.1.1 굴착 장비반입,전도,2020년 09월경 ○○현장에서 재해자가 굴삭기를 운전하여 화물자동차에서 내리던 중 넘어지는 굴삭기에...,굴착기(백호우),굴착기를 현장으로 반입하기 위해 화물차에서 굴착기를 내리는 중 전도되는 굴착기에 깔림,"▶ 화물자동차에서 발판 등을 사용하여 굴착기를 내리는 경우 충분한 길이, 폭, 강도를 가진것을 사용\..."
2,3,1. 토공사,1.1 굴착 작업,1.1.1 굴착 장비반입,부딪힘,2020년 03월경 ○○ 현장에서 굴삭기(0.6㎥)가 도로 측면 터파기 작업장소로 이동하기 위하여 약...,굴착기(백호우),굴착기가 도로 측면 터파기 작업장소로 이동 중 작업자와 충돌,▶ 차량계 건설기계 운행 경로와 별도로 작업자들이 이동할 수 있는 안전통로 확보\n▶ 건설기계에 근접...


[2_업종별규모별_재해자수]  30행 × 12열
컬럼: ['대업종', '구분', '5인 미만 근로자수', '5인 - 9인 근로자수', '10인 - 19인 근로자수', '20인 - 29인 근로자수', '30인 - 49인 근로자수', '50인 - 99인 근로자수', '100인 - 299인 근로자수', '300인 - 499인 근로자수', '500인 - 999인 근로자수', '1000인 이상 근로자수']


...

,대업종,구분,5인 미만 사업장수,5인 - 9인 사업장수,10인 - 19인 사업장수,20인 - 29인 사업장수,30인 - 49인 사업장수,50인 - 99인 사업장수,100인 - 299인 사업장수,300인 - 499인 사업장수,500인 - 999인 사업장수,1000인 이상사업장수
0,금융및보험업,금융및보험업,16174,9125,10682,4230,3241,1563,688,110,74,65
1,광 업,석탄광업및채석업,19,7,3,2,3,1,2,0,1,0
2,광 업,석회석·금속·비금속광업및기타광업,605,129,139,57,34,8,3,0,0,0


[5_업종별규모별_사망만인율]  30행 × 12열
컬럼: ['대업종', '구분', '5인 미만', '5인-9인', '10인-19인', '20인-29인', '30인-49인', '50인-99인', '100인-299인', '300인-499인', '500인-999인', '1000인 이상']


,대업종,구분,5인 미만,5인-9인,10인-19인,20인-29인,30인-49인,50인-99인,100인-299인,300인-499인,500인-999인,1000인 이상
0,금융및보험업,금융및보험업,1.77,0.63,0.28,0.30,0.00,0.10,0.10,0.23,0.00,0.17
1,광 업,석탄광업및채석업,4444.44,2272.73,2045.45,2156.86,1724.14,8333.33,2258.06,NaN,1642.44,NaN
2,광 업,석회석·금속·비금속광업및기타광업,95.60,47.23,10.27,14.74,15.41,110.91,80.21,NaN,NaN,NaN


[6_출처목록]  5행 × 4열
컬럼: ['구분', '개방url', '데이터규모', '주요설명']


,구분,개방url,데이터규모,주요설명
0,재해조사보고서,https://moel.go.kr/info/dsstExaminRpt/list.do,2024년 개시 단년도,"산업안전보건법 제56조의2, 같은법 시행규칙 제71조의2, 고용노동부는 국민의 알 권리 보장 및 산업..."
1,중대재해 사이렌,https://portal.kosha.or.kr/archive/imprtnDsstrAlrame/CSA...,"856건, 24~26년","중대재해 발생 알림(중대재해 사이렌)'은 사업장 관계자의 안전의식 및 경각심 고취를 통해 동종, 유사..."
2,업무상 사고사례,https://portal.kosha.or.kr/archive/disaster-case/acciden...,"1425건, 04~26년",산업안전포털을 통해 공개 제공중인 업무상 사고사례 데이터


## 4. 컬럼 타입과 비어 있지 않은 값 수 (info)

In [5]:
for name, d in dfs.items():
    print("=" * 90)
    print(f"[{name}]")
    d.info()

[1a_SIF_제조업등]
<class 'pandas.DataFrame'>
RangeIndex: 2573 entries, 0 to 2572
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   연번           2573 non-null   int64
 1   산재업종(대분류)    2573 non-null   str  
 2   산재업종(중분류)    2573 non-null   str  
 3   산재업종(소분류)    2573 non-null   str  
 4   재해개요         2573 non-null   str  
 5   기인물          2573 non-null   str  
 6   고위험작업·상황     2573 non-null   str  
 7   재해유발요인       2573 non-null   str  
 8   위험성감소대책(예시)  2573 non-null   str  
dtypes: int64(1), str(8)
memory usage: 181.0 KB
[1b_SIF_건설업]
<class 'pandas.DataFrame'>
RangeIndex: 3459 entries, 0 to 3458
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   연번           3459 non-null   int64
 1   공종           3459 non-null   str  
 2   작업명          3459 non-null   str  
 3   단위작업명        3459 non-null   str  
 4   재해종류         3459 non-null   str  
 5   재해개요      

## 5. 결측 · 중복 요약표

In [6]:
rows = []
for name, d in dfs.items():
    n_cell = d.shape[0] * d.shape[1]
    n_na = int(d.isna().sum().sum())
    rows.append({
        "파일": name,
        "행": d.shape[0],
        "열": d.shape[1],
        "결측셀": n_na,
        "결측비율(%)": round(n_na / n_cell * 100, 2) if n_cell else 0,
        "결측있는컬럼수": int((d.isna().sum() > 0).sum()),
        "완전중복행": int(d.duplicated().sum()),
    })
summary = pd.DataFrame(rows)
display(summary)

,파일,행,열,결측셀,결측비율(%),결측있는컬럼수,완전중복행
0,1a_SIF_제조업등,2573,9,0,0.00,0,0
1,1b_SIF_건설업,3459,9,0,0.00,0,0
2,2_업종별규모별_재해자수,30,12,0,0.00,0,0
3,3_규모별_사망자수,10,23,0,0.00,0,0
4,4_업종별규모별_사업장수,30,12,0,0.00,0,0
5,5_업종별규모별_사망만인율,30,12,14,3.89,4,0
6,6_출처목록,5,4,0,0.00,0,0


In [7]:
# 결측이 있는 컬럼만 파일별로 표시
for name, d in dfs.items():
    na = d.isna().sum()
    na = na[na > 0]
    print(f"[{name}]", "결측 없음" if na.empty else "")
    if not na.empty:
        display(pd.DataFrame({"결측수": na, "비율(%)": (na / len(d) * 100).round(1)}))

[1a_SIF_제조업등] 결측 없음
[1b_SIF_건설업] 결측 없음
[2_업종별규모별_재해자수] 결측 없음
[3_규모별_사망자수] 결측 없음
[4_업종별규모별_사업장수] 결측 없음
[5_업종별규모별_사망만인율] 


,결측수,비율(%)
100인-299인,1,3.3
300인-499인,3,10.0
500인-999인,2,6.7
1000인 이상,8,26.7


[6_출처목록] 결측 없음


In [8]:
# 1번 사례 데이터: 같은 재해개요가 반복되는지 확인 (삭제하지 않고 확인만)
for name in ["1a_SIF_제조업등", "1b_SIF_건설업"]:
    d = dfs[name]
    dup = d[d.duplicated("재해개요", keep=False) & d["재해개요"].notna()]
    print(f"[{name}] 재해개요가 같은 행: {len(dup)}개")
    if len(dup):
        display(dup.sort_values("재해개요")[["재해개요"]].head(10))

[1a_SIF_제조업등] 재해개요가 같은 행: 0개
[1b_SIF_건설업] 재해개요가 같은 행: 2개


,재해개요
2318,2019년 12월경 ○○현장에서 피재자가 이설도로의 콘크리트 포장작업을 위해 거리 측정을 도와주던 중...
2319,2019년 12월경 ○○현장에서 피재자가 이설도로의 콘크리트 포장작업을 위해 거리 측정을 도와주던 중...


## 6. 기준 연도 확인
- 1번 사례: `재해개요` 앞부분에 적힌 연도(예: `2023년 ...`)를 추출합니다.
- 3번 통계: 컬럼 이름에 들어 있는 연도 범위를 확인합니다.

In [9]:
YEAR = re.compile(r"(?<!\d)((?:19|20)\d{2})")

def case_years(s):
    y = s.dropna().astype(str).str.strip().str.extract(YEAR)[0].dropna().astype(int)
    return y

year_rows = []
for name in ["1a_SIF_제조업등", "1b_SIF_건설업"]:
    y = case_years(dfs[name]["재해개요"])
    year_rows.append({"대상": name + " (재해개요)", "연도 추출 건수": len(y),
                      "전체 대비(%)": round(len(y) / len(dfs[name]) * 100, 1),
                      "최소": y.min(), "최대": y.max()})
    print(f"[{name}] 연도별 건수")
    print(y.value_counts().sort_index().to_string(), "\n")

cols3 = [int(m.group(1)) for c in dfs["3_규모별_사망자수"].columns
         if (m := re.fullmatch(r"\s*((?:19|20)\d{2})\s*(?:년)?\s*", str(c)))]
if cols3:
    year_rows.append({"대상": "3번 통계 (컬럼)", "연도 추출 건수": len(cols3),
                      "전체 대비(%)": None, "최소": min(cols3), "최대": max(cols3)})
display(pd.DataFrame(year_rows))

[1a_SIF_제조업등] 연도별 건수
0
2014      4
2015     11
2016    352
2017    340
2018    349
2019    343
2020    307
2021    197
2022    347
2023    323 

[1b_SIF_건설업] 연도별 건수
0
2013      2
2014      2
2015     34
2016    500
2017    488
2018    433
2019    450
2020    442
2021    404
2022    371
2023    332
2024      1 



,대상,연도 추출 건수,전체 대비(%),최소,최대
0,1a_SIF_제조업등 (재해개요),2573,100.0,2014,2023
1,1b_SIF_건설업 (재해개요),3459,100.0,2013,2024


## 7. 핵심 컬럼 · 키 후보 확인
RAG 문서가 되는 1번 사례 데이터에서 각 컬럼이 얼마나 채워져 있고 값이 몇 종류인지 봅니다.
통계 파일(2~5번)은 `구분`(업종)과 규모 컬럼으로 서로 조인합니다.

In [10]:
for name in ["1a_SIF_제조업등", "1b_SIF_건설업"]:
    d = dfs[name]
    prof = pd.DataFrame({
        "채움(%)": (d.notna().mean() * 100).round(1),
        "고유값수": d.nunique(),
        "예시": d.apply(lambda s: str(s.dropna().iloc[0])[:40] if s.notna().any() else ""),
    })
    print(f"[{name}]")
    display(prof)

# 2~5번: 업종(구분) 컬럼과 값 확인
for name in ["2_업종별규모별_재해자수", "4_업종별규모별_사업장수", "5_업종별규모별_사망만인율"]:
    d = dfs[name]
    key = d.columns[0]
    print(f"[{name}] 첫 컬럼 '{key}' 값 {d[key].nunique()}종, 중복 {int(d[key].duplicated().sum())}개")
    print("  ", list(d[key].astype(str).head(8)), "...")

[1a_SIF_제조업등]


,채움(%),고유값수,예시
연번,100.0,2573,1
산재업종(대분류),100.0,9,제조업
산재업종(중분류),100.0,30,화학및고무제품제조업
산재업종(소분류),100.0,164,화학비료제조업
재해개요,100.0,2573,2019년 10월경 OO사업장에서 재해자가 화물용 리프트에 제품을 싣는
기인물,100.0,626,분류 불가
고위험작업·상황,100.0,37,리프트·승강기 관련 작업
재해유발요인,100.0,124,『화물용 승강기 또는 리프트(기계식 주차설비 포함)의 운반구(기)에서 작
위험성감소대책(예시),100.0,2529,▶ 개구부에 추락방지용 덮개 설치\n▶ 카(운반구)에 울 추가 설치\n▶ P


[1b_SIF_건설업]


,채움(%),고유값수,예시
연번,100.0,3459,1
공종,100.0,13,1. 토공사
작업명,100.0,49,1.1 굴착 작업
단위작업명,100.0,112,1.1.1 굴착 장비반입
재해종류,100.0,20,추락
재해개요,100.0,3458,2019년 03월경 ○○ 현장 1층에서 피재자가 장비(미니 백호 1ton
기인물,100.0,57,바닥개구부(자재인양구 등)
재해유발요인,100.0,3300,굴착기 진입을 위해 현장 정리 중 주차타워 상부 개구부를 인식하지 못하고
위험성감소대책(예시),100.0,1687,▶ 추락할 위험이 있는 개구부에는 안전난간 또는 덮개 등을 충분한 강도를


[2_업종별규모별_재해자수] 첫 컬럼 '대업종' 값 10종, 중복 20개
   ['금융및보험업', '광  업', '광  업', '제조업', '제조업', '제조업', '제조업', '제조업'] ...
[4_업종별규모별_사업장수] 첫 컬럼 '대업종' 값 10종, 중복 20개
   ['금융및보험업', '광  업', '광  업', '제조업', '제조업', '제조업', '제조업', '제조업'] ...
[5_업종별규모별_사망만인율] 첫 컬럼 '대업종' 값 10종, 중복 20개
   ['금융및보험업', '광  업', '광  업', '제조업', '제조업', '제조업', '제조업', '제조업'] ...


## 8. 데이터 명세 초안 (저장)

In [11]:
spec = pd.DataFrame([
    ["1a_SIF_제조업등", "산재 사고사례(제조업 등)", "RAG 문서", "연번", "재해개요에서 연도 추출 필요"],
    ["1b_SIF_건설업",   "산재 사고사례(건설업)",     "RAG 문서", "연번", "재해개요에서 연도 추출 필요"],
    ["2_업종별규모별_재해자수",   "업종 × 규모 통계", "통계/차트", "구분(업종)", "업종명 표기 통일 필요"],
    ["3_규모별_사망자수",         "규모 × 연도 통계", "통계/차트", "규모",       "연도가 컬럼으로 펼쳐진 형태"],
    ["4_업종별규모별_사업장수",   "업종 × 규모 통계", "통계/차트", "구분(업종)", "사망만인율 해석에 사용"],
    ["5_업종별규모별_사망만인율", "업종 × 규모 통계", "통계/차트", "구분(업종)", "사업장수 0인 칸의 결측은 의미 있는 결측"],
    ["6_출처목록",     "데이터 출처 주소 목록", "참고(분석 제외)", "-", "사례 데이터가 아니라 출처 목록"],
], columns=["파일", "내용", "용도", "키 컬럼", "비고"])
display(spec)

OUT = ROOT / "data" / "analysis"
OUT.mkdir(parents=True, exist_ok=True)
result = summary.merge(spec, on="파일", how="left")
result.to_csv(OUT / "01_data_check_summary.csv", index=False, encoding="utf-8-sig")
print("저장:", (OUT / "01_data_check_summary.csv").relative_to(ROOT))

,파일,내용,용도,키 컬럼,비고
0,1a_SIF_제조업등,산재 사고사례(제조업 등),RAG 문서,연번,재해개요에서 연도 추출 필요
1,1b_SIF_건설업,산재 사고사례(건설업),RAG 문서,연번,재해개요에서 연도 추출 필요
2,2_업종별규모별_재해자수,업종 × 규모 통계,통계/차트,구분(업종),업종명 표기 통일 필요
3,3_규모별_사망자수,규모 × 연도 통계,통계/차트,규모,연도가 컬럼으로 펼쳐진 형태
4,4_업종별규모별_사업장수,업종 × 규모 통계,통계/차트,구분(업종),사망만인율 해석에 사용
5,5_업종별규모별_사망만인율,업종 × 규모 통계,통계/차트,구분(업종),사업장수 0인 칸의 결측은 의미 있는 결측
6,6_출처목록,데이터 출처 주소 목록,참고(분석 제외),-,사례 데이터가 아니라 출처 목록


저장: data/analysis/01_data_check_summary.csv


## 9. 결과 메모 (직접 채우기)
실행 결과를 보고 아래를 확인·수정해 두면 M2 전처리 기준이 됩니다.
- 결측이 많은 컬럼과 처리 방침 (삭제하지 않고 NaN 유지가 기본)
- 중복 행 / 같은 재해개요 사례 처리 방침
- 기준 연도 범위와 연도 추출이 안 되는 사례 수
- 1번(RAG 문서)과 2~5번(통계)의 역할 구분